# PokouAI — Quantize Gemma 4 fine-tune to GGUF

Takes the merged HF model from `03_finetune_cocoa_unsloth.ipynb` (cell 18, `model.save_pretrained_merged(...)`) and produces the two artefacts the app sideloads:

- `cocoa_v1_<variant>.gguf` — quantized main model (Q4_K_M default; Q3_K_M for tight RAM)
- `cocoa_v1_<variant>-mmproj.gguf` — F16 vision projector

**Why a separate notebook**: Unsloth's bundled llama.cpp converter fails on Gemma 4's multimodal vision tower (`unsloth_convert_hf_to_gguf.py --mmproj` returns non-zero). We build a fresh llama.cpp HEAD here, which handles Gemma 4 correctly.

**Prereq**: notebook 03 has run through cell 18 in this Kaggle session, leaving the merged HF dir at `/kaggle/working/cocoa_v1_<variant>_merged/`.

**Disk**: build (~250 MB) + F16 (~10 GB) + final GGUF (~3 GB) + mmproj (~880 MB) + merged source (~5 GB) ≈ 19 GB peak. /kaggle/working has 20 GB. F16 is deleted after quantization.

## 0 — Parameters

In [ ]:
VARIANT = 'e2b'      # 'e2b' or 'e4b' — must match notebook 03
QUANT   = 'Q3_K_M'   # Q4_K_M (~3.4 GB E2B) or Q3_K_M (~2.5 GB) — Q3 fits iPhone jetsam with mmproj loaded

import os, subprocess
MERGED_DIR  = f'/kaggle/working/cocoa_v1_{VARIANT}_merged'
OUTDIR      = '/kaggle/working'
F16_PATH    = f'{OUTDIR}/cocoa_v1_{VARIANT}.F16.gguf'
OUT_PATH    = f'{OUTDIR}/cocoa_v1_{VARIANT}.gguf'
MMPROJ_PATH = f'{OUTDIR}/cocoa_v1_{VARIANT}-mmproj.gguf'

assert os.path.isdir(MERGED_DIR), f'missing {MERGED_DIR} — re-run notebook 03 through cell 18 first'
print(f'merged source: {MERGED_DIR}')
print(f'target quant : {QUANT}')
!df -h /kaggle/working

## 1 — Build llama.cpp
Clone HEAD and build the two binaries we need: `llama-quantize` (CLI) and `convert_hf_to_gguf.py` (Python). CUDA off — we don't need GPU for quantization. Takes ~3 min on Kaggle CPU.

In [ ]:
!rm -rf /kaggle/working/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DBUILD_SHARED_LIBS=OFF 2>&1 | tail -5
!cd /kaggle/working/llama.cpp && cmake --build build --target llama-quantize llama-cli -j$(nproc) 2>&1 | tail -10
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!ls -la /kaggle/working/llama.cpp/build/bin/llama-quantize

## 2 — Convert merged HF → F16 GGUF (main model)
Lossless conversion. Output is ~10 GB for E2B — intermediate file, deleted after quantization.

In [ ]:
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {F16_PATH} --outtype f16
!ls -lh {F16_PATH}

## 3 — Quantize F16 → target
F16 is deleted right after to free disk for the mmproj export.

In [ ]:
!/kaggle/working/llama.cpp/build/bin/llama-quantize {F16_PATH} {OUT_PATH} {QUANT}
!ls -lh {OUT_PATH}
!rm -f {F16_PATH}
!df -h /kaggle/working

## 4 — Vision projector (mmproj)
Fresh llama.cpp's `--mmproj` flag extracts the vision tower as a standalone GGUF. We keep it at F16 — quantizing the vision encoder hurts disproportionately, and 880 MB is fine on modern phones.

In [ ]:
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {MMPROJ_PATH} --mmproj
!ls -lh {MMPROJ_PATH}

## 5 — Verify + report

In [ ]:
for p in [OUT_PATH, MMPROJ_PATH]:
    sz = os.path.getsize(p) / 1e9
    print(f'{os.path.basename(p):42s} {sz:5.2f} GB')

# Quick GGUF header check — confirms the file isn't truncated
print('\n--- main GGUF metadata ---')
!/kaggle/working/llama.cpp/build/bin/llama-cli -m {OUT_PATH} --version 2>&1 | head -5 || true

print(f'\nDownload from Kaggle output → sideload via Finder → iPhone → Files → PokouAI:')
print(f'  {OUT_PATH}')
print(f'  {MMPROJ_PATH}')

## 6 — Optional: push to HF Hub
Useful for the final submission so the app can auto-download instead of requiring a Finder sideload. Skip during iteration.

In [ ]:
# from huggingface_hub import HfApi, login
# login(token='hf_...')
# REPO = f'<your-username>/pokouai-cocoa-{VARIANT}'
# api = HfApi()
# api.create_repo(REPO, exist_ok=True)
# for path in [OUT_PATH, MMPROJ_PATH]:
#     api.upload_file(path_or_fileobj=path, path_in_repo=os.path.basename(path), repo_id=REPO)
# print(f'pushed to https://huggingface.co/{REPO}')